# 4 · Tables

Every level of a graph is a table. A node that returns many values makes a table, and a node that runs once per row adds a column to the table it ran over. A `Table` input or output is a `polars.DataFrame`.

In [ ]:
from typing import TypedDict

import polars as pl
from conductor import CompiledGraph, From, Graph, GraphNode, NodeRegistry, Series, Table, node


class Document(TypedDict):
    name: str


@node("documents")
def documents() -> Table[Document]:
    return pl.DataFrame({"name": ["a.pdf", "b.pdf"]})


@node("lines")
def lines(name: str) -> Series[str]:
    return ["first line", "second line"] if name == "a.pdf" else []


@node("shout")
def shout(line: str) -> str:
    return line.upper()


@node("count")
def count(values: Series[str]) -> int:
    return len(values)


@node("sheet")
def sheet(rows: Table) -> Table:
    return rows


registry = NodeRegistry([documents, lines, shout, count, sheet])

## Wiring decides how a node runs

`documents.result.name` is one column of the table. `lines` takes one value, so it runs once per document and makes rows under each. `count` takes a `Series`, so it gets every line under its document. `sheet` takes a `Table`, and what's wired into it is merged.

In [ ]:
graph = Graph(nodes=[
    documents.graph_node("documents"),
    lines.graph_node("lines", name=From("documents.result.name")),
    shout.graph_node("shout", line=From("lines.result")),
    count.graph_node("how_many", values=From("lines.result")),
    GraphNode(id="per_line", type="sheet", version=1, bindings={"rows": From("documents.result.name", "shout.result")}),
    GraphNode(id="per_document", type="sheet", version=1, bindings={"rows": From("documents.result.name", "how_many.result")}),
])
compiled = CompiledGraph.from_graph(graph, registry)
print(compiled.run_plan)

## Merging

Rows are matched on the hidden `__row` column. Beside its lines, a document's name is copied down, so that sheet has one row per line and `b.pdf`, which has no lines, has no row in it. The per-document sheet has one row per document, so `b.pdf` shows with a count of 0.

In [ ]:
results = compiled.results(await compiled.run())
print(results["per_line"]["result"])
print(results["per_document"]["result"])

## Blanks

A blank cell reaches a node only if its input says `| None`. Otherwise the node isn't called for that row, its output is blank there, and the state counts it.

In [ ]:
class Contact(TypedDict):
    email: str


@node("contacts")
def contacts() -> Table[Contact]:
    return pl.DataFrame({"email": ["ida@a.dk", None]}, schema={"email": pl.String})


@node("domain")
def domain(email: str) -> str:
    return email.split("@")[1]


@node("has-email")
def has_email(email: str | None) -> bool:
    return email is not None


blanks = CompiledGraph.from_graph(
    Graph(nodes=[
        contacts.graph_node("contacts"),
        domain.graph_node("domain", email=From("contacts.result.email")),
        has_email.graph_node("has_email", email=From("contacts.result.email")),
    ]),
    NodeRegistry([contacts, domain, has_email]),
)
state = await blanks.run()
results = blanks.results(state)
print(results["domain"]["result"]["values"], results["has_email"]["result"]["values"])
print({node_id: work.rows_not_called for node_id, work in state.nodes.items() if work.rows_not_called})